# Lexorion — DeBERTa fine-tuning on a free Colab GPU

Fine-tunes **one multi-label DeBERTa-v3-base** (8 sigmoid heads, one per risk category)
on CUAD paragraphs, with the project's recall-first thresholding policy.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Total runtime:** ~50-70 min (data rebuild ~10 min, training ~35-50 min).
The last cell downloads the two result files you carry back to the repo —
the 700MB model checkpoint stays here unless you want it.

In [1]:
# 1. Confirm the GPU is on (should print a Tesla T4 table)
!nvidia-smi

Mon Jul  6 23:00:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Get the code
!git clone --depth 1 https://github.com/DhavalVibhakar99/Lexorion-Cuad.git
%cd Lexorion-Cuad

Cloning into 'Lexorion-Cuad'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 69 (delta 0), reused 63 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 142.47 KiB | 8.90 MiB/s, done.
/content/Lexorion-Cuad


In [3]:
# 3. Dependencies (Colab already ships torch + CUDA)
%pip install -q transformers accelerate datasets sentencepiece protobuf pyyaml pyarrow

In [4]:
# 4. Rebuild the CUAD paragraph dataset from the public source (~10 min)
!python -m src.data_pipeline.download_cuad
!python -m src.data_pipeline.parse_cuad
!python -m src.data_pipeline.chunk_contracts

Download complete.
Extracting...
Extraction complete.
  Found: train_separate_questions.json
  Found: test.json
  Found: CUADv1.json

Parsing train data...
Parsing test data...

Dataset loaded successfully!
  Train split: 22450 examples
  Test split:  4182 examples
Saving the dataset (3/3 shards): 100% 22450/22450 [00:20<00:00, 1114.75 examples/s]
Saving the dataset (1/1 shards): 100% 4182/4182 [00:03<00:00, 1160.32 examples/s]

Saved HuggingFace dataset to data/raw/cuad_hf

Extracted 408 unique contracts
Saved contract index to data/raw/contract_index.json

CUAD Dataset Summary
  Total contracts:        408
  Total QA pairs:         22450
  Pairs with answers:     11180
  Pairs without answers:  11270
  Answer density:         49.8%
Loading CUAD dataset...

PARSED DATASET SUMMARY
Total QA pairs:           22,450
Unique contracts:         408
Unique CUAD categories:   41
Pairs with clauses:       11,180 (49.8%)
Mapped to risk categories:13,881

--- Clause counts per Risk Category ---
 

In [7]:
%pip install -q "transformers<5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [8]:
# 5. Train (single multi-label run: BCE + per-class pos_weight, fp16, 256 tokens)
#    Prints the per-category recall-first results table at the end.
!python -m src.models.clause_detector --epochs 2 --batch_size 16


2026-07-06 23:06:25.083121: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Training microsoft/deberta-v3-base on 25,759 paragraphs, 8 categories
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/d

In [9]:
# 6. Download the two result files to carry back to the repo
from google.colab import files
files.download('data/processed/deberta_predictions.parquet')
files.download('data/processed/deberta_training_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>